# featurewiz, explained 3/4: Recursive XGBoost, step by step
The second stage takes what SULOV kept and asks XGBoost which features it actually uses.
It doesn't ask once. It runs several **Rounds**, each on a shrinking **window** of columns,
and pools every Round's top features.

In [ ]:
import pathlib, sys

for module_file in pathlib.Path("/kaggle/input").glob("**/featurewiz_explained.py"):
    sys.path.insert(0, str(module_file.parent))
    break

import featurewiz_explained as fwx
import featurewiz_explained_plots as plots

> **featurewiz, explained**: a four-part series that rebuilds featurewiz 0.6.1's feature selection step by step.
> 1 Intuition · 2 SULOV · **3 Recursive XGBoost** · 4 Full pipeline.
> Everything here runs with Internet **off**. Results are checked against Reference runs of the real featurewiz 0.6.1.

In [ ]:
data = fwx.make_canonical_dataset()
target = data.target
features = fwx.sulov(data.frame, data.features, target).kept  # SULOV's output order, as featurewiz passes it on
features

## Step 1: plan the Rounds
- **step**: how many columns each Round slides past (2 if there are ≤ 10 features, otherwise about n/5).
- **top_num**: how many features each Round nominates at least (25% of n if n ≤ 50, otherwise 20%).

In [ ]:
n = len(features)
step = 2 if n <= 10 else int(n / 5 + 0.5)
top_num = int(max(2, n * 0.25)) if n <= 50 else int(n * 0.20)
assert (step, top_num) == fwx.round_plan(n)
print(f"{n} features -> step {step}, top {top_num} per Round")

## Step 2: run the Rounds
Round *k* trains on `features[k*step:]`, a **suffix**. The window drops columns from the *front*, so the
first columns are seen in one Round and the last ones in every Round. Hold on to that.

Which features does a Round nominate? Those with at least **half the top feature's gain**, unless that
gives fewer than `top_num`, in which case it takes the top `top_num` by gain.

In [ ]:
import pandas as pd
import xgboost as xgb

params = {"objective": "binary:logistic", "num_class": 1, "min_child_weight": 0.5, "tree_method": "exact"}
selected, rounds = [], []
for i in range(0, n, step):
    window = features[i:]
    if len(window) < 2:
        continue
    booster = xgb.train(params, xgb.DMatrix(data.frame[window], data.frame[target], feature_names=window),
                        num_boost_round=100)
    gains = booster.get_score(importance_type="total_gain")
    nominated = fwx.nominate(gains, top_num)
    selected = list(dict.fromkeys(selected + nominated))
    rounds.append(window)
    print(f"Round {len(rounds)}: window starts at {window[0]!r:14} nominates {nominated}")
selected

In [ ]:
result = fwx.recursive_xgboost(data.frame, features, target)
assert selected == result.selected
plots.rounds_heatmap(result).show()

**Why was `x4` missed?** `x4` and `x4_squared` sit at the *front* of the column order (SULOV puts unpaired
features first), so they're only in Round 1. There, `x1`, `x2` and `x3_copy` all clear "half of the top gain",
so that's three nominations, more than `top_num = 2`, and `x4` isn't one of them. In later Rounds `x4` isn't
in the window at all.

## The Quirk: shuffle the columns, change the answer
Same data, same features, same model. Only the column order differs:

In [ ]:
shuffled = fwx.shuffled_order(features, seed=0)
shuffled_result = fwx.recursive_xgboost(data.frame, shuffled, target)
print("order:   ", shuffled)
print("selected:", shuffled_result.selected)
plots.rounds_heatmap(shuffled_result).show()

Now `x4` is found, along with `x4_squared` and `noise_2`. The real featurewiz does the same. In the pinned
environment (xgboost 1.7.6) the match is exact down to every gain value. On a newer xgboost the gains drift a
little, so `exact` can print `False` here even when the selection is identical; `selected_matches` is the
part to look at.

In [ ]:
reference = fwx.load_reference("canonical_shuffled")
print("same column order as the Reference run:", reference.input_features == shuffled)
comparison = fwx.compare_to_reference(shuffled_result, reference)
print("featurewiz 0.6.1 selected:", reference.selected)
print("selection matches:", comparison.selected_matches, "| exact to the last gain:", comparison.exact)

## A version note: why `tree_method="exact"`?
The Reference runs used xgboost 1.7.6, where small datasets are trained with the *exact* tree method.
xgboost 2.0 changed the default to *hist*. If you don't pin it, gains shift, and sometimes nominations do too:

In [ ]:
import sklearn

print("this notebook: xgboost", xgb.__version__, "scikit-learn", sklearn.__version__)
print("reference runs:", reference.versions)
print("pinned environment:", fwx.matches_reference_environment(reference))
default_result = fwx.recursive_xgboost(data.frame, features, target, tree_method=None)
print("with this xgboost's default tree method:", default_result.selected)

With xgboost 3.4's default (*hist*) the same data adds `noise_3` to the selection: one tree-method switch,
one extra noise feature. If you run featurewiz in production, pin xgboost.

## What to remember
- Each Round sees a **suffix** of the columns, so **column order changes the result**.
- Every Round nominates **at least `top_num`** features, even when they're noise.
- Results depend on the xgboost version. Pin it if you need reproducibility.